# GEO-Bench segmentation feature selection

This notebook documents the completed dense-representation ablation for frozen semantic-segmentation probing. The repository protocol is now fixed to `final_pre_norm` for both `m-cashew-plant` and `m-SA-crop-type`; test results never select the representation.

The final headline evaluation is separate: a frozen backbone with a UPerNet decoder trained for 50 epochs. This notebook uses a shared 1x1 pixel probe so representation selection is fast and every candidate has exactly the same trainable capacity.

## 1. Context and methods

### Dense candidates

| Candidate | Spatial representation | Width |
|---|---|---:|
| `multilayer_pre_norm` | Raw fine-token maps from four encoder depths | D per stage |
| `multilayer_post_norm` | The same maps after the frozen final LayerNorm | D per stage |
| `final_pre_norm` | Raw final fine-token map repeated across the four decoder inputs | D per stage |
| `final_post_norm` | Final normalized fine-token map repeated across the four inputs | D per stage |

The shared 1x1 classifier is applied to every stage and its four logit maps are averaged before bilinear upsampling. CLS, metadata tokens and scene means are excluded because they discard the spatial correspondence required for pixel labels.

The encoder is frozen in evaluation mode and routing is deterministic. Inputs use official GEO-Bench per-band z-score statistics, with raw zero and non-finite values treated as nodata. The frozen headline protocol resizes encoder inputs to the checkpoint size and evaluates labels and predictions at 224 x 224. Native-resolution input is a separate spatial ablation. Each run keeps the epoch with the highest validation mean IoU.

In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

default_run_dir = repo_root
run_dir = Path(os.environ.get('MEOX_RUN_DIR', default_run_dir))
config_path = Path(os.environ.get(
    'MEOX_CONFIG', repo_root / 'configs/pretrain_mmearth_moe_mae_full.yaml'
))
checkpoint_path = Path(os.environ.get(
    'MEOX_CHECKPOINT', repo_root / 'weights/pretrained/meox_s_mmearth64_best.pth'
))
geobench_root = Path(os.environ['GEO_BENCH_DIR'])
cache_dir = run_dir / 'analysis/geobench_segmentation_selection'


dataset_names = ('m-cashew-plant', 'm-SA-crop-type')
learning_rates = {'m-cashew-plant': 3e-4, 'm-SA-crop-type': 1e-2}
candidates = (
    'multilayer_pre_norm', 'multilayer_post_norm',
    'final_pre_norm', 'final_post_norm',
)
selected_candidate = 'final_pre_norm'

input_image_size = None  # None resolves to model.img_size below.
label_image_size = 224
extraction_batch_size = 8
probe_batch_size = 16
num_workers = 4
probe_epochs = 50
probe_seeds = (0, 1, 2, 3, 4)
max_samples = None
sample_seed = 42
reuse_dense_features = True
evaluate_selected_on_test = True

### Optional smoke settings

Uncomment these assignments for a code-path check. Smoke results are not scientific results and must not change the frozen representation protocol.

In [ ]:
# max_samples = 32
# probe_epochs = 2
# probe_seeds = (0,)
# extraction_batch_size = 4
# probe_batch_size = 4
# num_workers = 0
# evaluate_selected_on_test = False

In [ ]:
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(repo_root))

from datasets.geobench import GeoBenchSegmentationDataset
from utils.extract_embeddings import (
    build_model_from_config, load_config, make_inference_dataloader,
    maybe_limit_dataset, model_input_schema, resolve_device,
)
from utils.segmentation_probe import (
    dense_cache_matches, dense_extraction_signature,
    extract_dense_feature_cache, resolve_feature_layers,
    train_segmentation_probe,
)

## 2. Frozen model and official splits

In [ ]:
device = resolve_device()
config = load_config(str(config_path))
if input_image_size is None:
    input_image_size = int(config['model']['img_size'])
_, model_band_names, _ = model_input_schema(config)
model = build_model_from_config(config, str(checkpoint_path), device)
feature_layers = resolve_feature_layers(len(model.encoder.layers))
cache_dir.mkdir(parents=True, exist_ok=True)

print('Device:', device)
print('Encoder layers:', len(model.encoder.layers))
print('Dense feature layers:', feature_layers)
print('Candidates:', candidates)

In [ ]:
def prepare_split_cache(dataset_name, split):
    dataset = GeoBenchSegmentationDataset(
        root_dir=geobench_root, dataset_name=dataset_name, split=split,
        model_band_names=model_band_names, input_image_size=input_image_size,
        label_image_size=label_image_size,
    )
    extraction_dataset = maybe_limit_dataset(dataset, max_samples, sample_seed)
    dataset_cache_dir = cache_dir / dataset_name
    dataset_cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = dataset_cache_dir / f'{split}_dense_features.h5'
    signature = dense_extraction_signature(
        checkpoint_path=checkpoint_path,
        preprocessing_signature=dataset.preprocessing_signature,
        raster_band_names=dataset.raster_band_names,
        feature_layers=feature_layers, split=split, partition='default',
        max_samples=max_samples, sample_seed=sample_seed,
    )
    if not (reuse_dense_features and dense_cache_matches(cache_path, signature)):
        dataloader = make_inference_dataloader(
            extraction_dataset, extraction_batch_size, num_workers
        )
        extract_dense_feature_cache(
            model, dataloader, dataset.raster_band_names, device, feature_layers,
            cache_path, signature,
        )
    return cache_path, dataset

train_caches, valid_caches, dataset_info = {}, {}, {}
for dataset_name in dataset_names:
    train_caches[dataset_name], dataset_info[dataset_name] = prepare_split_cache(
        dataset_name, 'train'
    )
    valid_caches[dataset_name], _ = prepare_split_cache(dataset_name, 'valid')
    print(dataset_name, 'classes:', dataset_info[dataset_name].num_classes)

## 3. Validation-only dense feature ablation

Every candidate uses the same frozen inputs, cached encoder activations, shared 1x1 head, optimizer, epochs and seeds. Only the dense representation changes.

In [ ]:
candidate_results = {}
validation_rows = []
for dataset_name in dataset_names:
    for candidate in candidates:
        print(f'Running {dataset_name} / {candidate}')
        result = train_segmentation_probe(
            train_cache=train_caches[dataset_name],
            val_cache=valid_caches[dataset_name],
            num_classes=dataset_info[dataset_name].num_classes,
            candidate=candidate,
            norm_weight=model.encoder.norm.weight,
            norm_bias=model.encoder.norm.bias,
            learning_rate=learning_rates[dataset_name],
            device=device, head_type='linear', epochs=probe_epochs,
            batch_size=probe_batch_size, num_workers=num_workers,
            seeds=probe_seeds,
        )
        candidate_results[(dataset_name, candidate)] = result
        aggregate = result['aggregate_validation']['mean_iou']
        validation_rows.append({
            'dataset': dataset_name, 'candidate': candidate,
            'mean_validation_miou': aggregate['mean'],
            'std_validation_miou': aggregate['std'],
        })

validation_table = pd.DataFrame(validation_rows)
display(validation_table.round(4))

## 4. Freeze the candidate

The completed ablation had the best cross-dataset validation mean for `final_pre_norm`. The protocol is therefore fixed explicitly rather than using a tolerance or priority rule.

In [ ]:
selection_table = (
    validation_table.groupby('candidate', as_index=False)
    .agg(mean_validation_miou=('mean_validation_miou', 'mean'))
)
display(selection_table.sort_values('mean_validation_miou', ascending=False).round(4))
print('Frozen candidate:', selected_candidate)

figure, axis = plt.subplots(figsize=(8, 4))
ordered = selection_table.sort_values('mean_validation_miou')
axis.barh(ordered['candidate'], 100 * ordered['mean_validation_miou'], color='#176B87')
axis.set_xlabel('Mean validation mIoU across tasks (%)')
axis.set_title('Dense segmentation representation selection')
plt.tight_layout()
plt.show()

## 5. Final linear-probe test estimate

Only this section creates test caches. It retrains the already selected linear probe with the same deterministic seeds and reports test mIoU. Test performance must not change `selected_candidate`.

In [ ]:
test_results = {}
test_rows = []
if evaluate_selected_on_test:
    for dataset_name in dataset_names:
        test_cache, _ = prepare_split_cache(dataset_name, 'test')
        result = train_segmentation_probe(
            train_cache=train_caches[dataset_name],
            val_cache=valid_caches[dataset_name],
            test_cache=test_cache,
            num_classes=dataset_info[dataset_name].num_classes,
            candidate=selected_candidate,
            norm_weight=model.encoder.norm.weight,
            norm_bias=model.encoder.norm.bias,
            learning_rate=learning_rates[dataset_name],
            device=device, head_type='linear', epochs=probe_epochs,
            batch_size=probe_batch_size, num_workers=num_workers,
            seeds=probe_seeds,
        )
        test_results[dataset_name] = result
        aggregate = result['aggregate_test']['mean_iou']
        test_rows.append({
            'dataset': dataset_name, 'candidate': selected_candidate,
            'mean_test_miou': aggregate['mean'],
            'std_test_miou': aggregate['std'],
        })
    display(pd.DataFrame(test_rows).round(4))

    manifest = {
        'checkpoint': str(checkpoint_path.resolve()),
        'feature_layers': list(feature_layers),
        'selected_candidate': selected_candidate,
        'selection_metric': 'frozen_validation_protocol',
        'input_image_size': input_image_size,
        'label_image_size': label_image_size,
        'probe_epochs': probe_epochs,
        'probe_seeds': list(probe_seeds),
        'validation': validation_rows,
        'test': test_rows,
    }
    with open(cache_dir / 'selected_dense_feature.json', 'w', encoding='utf-8') as handle:
        json.dump(manifest, handle, indent=2)
else:
    print('Test evaluation disabled. Selection is complete:', selected_candidate)

## 6. Interpretation and next step

- `multilayer` beating `final` means intermediate spatial representations add information that the last layer loses.
- `pre_norm` beating `post_norm` means the final LayerNorm removes magnitude or direction information useful for pixel classification.
- `final_pre_norm` is frozen for all headline segmentation runs; do not select a candidate per dataset.
- The linear-probe mIoU is a representation diagnostic, not the CSMoE-comparable headline number.

The evaluation runner defaults to `final_pre_norm`, freezes the same backbone, and trains UPerNet for 50 epochs using the task-specific learning rates. Do not rerun or expand this ablation for the headline results.

In [ ]:
print(
    'nohup python -u evaluate_geobench_segmentation.py '
    f'--config_yaml {config_path} --checkpoint_path {checkpoint_path} '
    f'--data_root {geobench_root} --output_dir outputs/geobench_segmentation '
    f'--feature_candidate {selected_candidate} --reuse_dense_features '
    '--seeds 0 1 2 3 4 > geobench_segmentation.out 2>&1 &'
)